# Historical Spelling and Orthographic Normalisation

**Computational Text Analysis Pack · Module 14 — Preparing Text, Properly**

*Technique:* Edit distance, phonetic matching and rewrite rules; the VARD/Norma approach

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/01-preparing-text/14c_historical_normalisation.ipynb)

Data: `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

Every frequency count in this course assumes that one spelling means one word.
Historical text breaks that assumption: *shew* and *show*, *publick* and
*public*, *connexion* and *connection* are the same word counted twice, and your
keyness list quietly splits in half.

The standard tools are **VARD2** (Baron and Rayson, Lancaster) and **Norma**
(Bollmann, Bochum). Both combine the same four ingredients, and all four are
implementable here: a known-variant list, letter-rewrite rules, edit distance,
and phonetic matching.

## The tooling

VARD2 is Java, interactive, and needs training on samples of your corpus; Norma
is modular and (semi-)automatic. Bollmann's 2019 comparison across eight
languages found character-based statistical MT usually best but **no single
method optimal for all languages** — so the sensible design is a pipeline of
cheap methods with a confidence score, which is what we build below. Both tools
are mature and no longer heavily developed; verify before depending on them.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
counts = Counter()
for t in df_colonial["text"]:
    counts.update(tokenise(t))
print(f"{sum(counts.values()):,} tokens, {len(counts):,} distinct forms")

# A modern lexicon proxy: forms frequent enough to be safely modern.
MODERN = {w for w, n in counts.items() if n >= 40}
candidates = [w for w, n in counts.items() if 3 <= n < 40 and w.isalpha()]
print(f"{len(MODERN):,} forms treated as modern, {len(candidates):,} to check")

## Ingredient 1 — rewrite rules

A handful of letter substitutions covers a large share of Early Modern and
nineteenth-century variation. These are rules a scholar of the period writes,
not statistics.

In [ ]:
RULES = [("ck", "c"), ("ie", "y"), ("oe", "e"), ("ae", "e"),
         ("xion", "ction"), ("our", "or"), ("ise", "ize"),
         ("shew", "show"), ("ll", "l"), ("tt", "t"), ("ee", "e")]

def by_rules(word):
    out = set()
    for a, b in RULES:
        if a in word:
            out.add(word.replace(a, b))
        if b in word:
            out.add(word.replace(b, a))
    return {w for w in out if w in MODERN and w != word}

hits = {w: by_rules(w) for w in candidates}
hits = {w: v for w, v in hits.items() if v}
print(f"rules alone map {len(hits)} variants. A sample:")
for w in sorted(hits, key=lambda w: -counts[w])[:12]:
    print(f"  {w:<18}({counts[w]:>3}x) -> {', '.join(sorted(hits[w]))}")

## Ingredients 2 and 3 — edit distance and phonetic matching

Edit distance catches variation the rules missed. Phonetic matching, in the
SoundEx family VARD2 uses, catches variation that *sounds* the same but looks
different — which is most of what a scribe or compositor actually produced.

In [ ]:
def levenshtein(a, b, cap=2):
    if abs(len(a) - len(b)) > cap:
        return cap + 1
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        if min(cur) > cap:
            return cap + 1
        prev = cur
    return prev[-1]

def soundex(word):
    codes = {**dict.fromkeys("bfpv", "1"), **dict.fromkeys("cgjkqsxz", "2"),
             **dict.fromkeys("dt", "3"), "l": "4",
             **dict.fromkeys("mn", "5"), "r": "6"}
    if not word:
        return ""
    first, out, prev = word[0].upper(), "", codes.get(word[0], "")
    for ch in word[1:]:
        c = codes.get(ch, "")
        if c and c != prev:
            out += c
        if ch not in "hw":
            prev = c
    return (first + out + "000")[:4]

modern_by_sound = {}
for w in MODERN:
    modern_by_sound.setdefault(soundex(w), []).append(w)

def normalise(word):
    if word in MODERN:
        return word, "already modern", 1.0
    r = by_rules(word)
    if r:
        return max(r, key=lambda w: counts[w]), "rewrite rule", 0.9
    same_sound = [w for w in modern_by_sound.get(soundex(word), [])
                  if levenshtein(word, w) <= 2]
    if same_sound:
        return max(same_sound, key=lambda w: counts[w]), "phonetic + edit", 0.6
    near = [w for w in MODERN if levenshtein(word, w) <= 1]
    if near:
        return max(near, key=lambda w: counts[w]), "edit distance 1", 0.5
    return word, "unresolved", 0.0

sample = sorted(candidates, key=lambda w: -counts[w])[:400]
results = [(w, *normalise(w)) for w in sample]
res = pd.DataFrame(results, columns=["variant", "normalised", "method", "confidence"])
print(res["method"].value_counts().to_string())

In [ ]:
changed = res[(res["variant"] != res["normalised"])]
print(f"{len(changed)} of {len(res)} sampled forms were normalised:\n")
print(changed.sort_values("confidence", ascending=False)
      .head(20).to_string(index=False))
print()
print("Read that list critically. Some are right; some collapse genuinely")
print("different words. VARD2 is interactive for exactly this reason — the")
print("scholar confirms, the tool proposes.")

## What normalisation costs and buys

Normalising changes every count downstream. That is the point, and it is also
the risk, so it should be measured rather than assumed.

In [ ]:
mapping = {v: n for v, n, m, c in results if v != n and c >= 0.6}
before = Counter(counts)
after = Counter()
for w, n in counts.items():
    after[mapping.get(w, w)] += n

print(f"distinct forms before : {len(before):,}")
print(f"distinct forms after  : {len(after):,}")
print(f"reduction             : {1 - len(after)/len(before):.2%}")
print()
moved = sorted(mapping.items(), key=lambda kv: -counts[kv[0]])[:10]
print(f"{'form':<16}{'was':>7}{'merged into':>16}{'now':>9}")
print("-" * 48)
for v, n in moved:
    print(f"{v:<16}{before[v]:>7}{n:>16}{after[n]:>9}")

plt.figure(figsize=(6.5, 3.6))
plt.bar(["before", "after"], [len(before), len(after)],
        color=["steelblue", "darkorange"])
plt.ylabel("distinct forms")
plt.title("Normalisation shrinks the vocabulary")
plt.tight_layout()

### 🔧 Your turn 1

Set the confidence cut to 0.5 so edit-distance-1 matches are included. How many more merges happen, and are they safe?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Dropping to 0.5 roughly doubles the merges and introduces clear errors: at edit
distance 1, *hat/hate*, *of/on* and many proper nouns collapse into each other.
This is the reason VARD2 ranks candidates by a combined F-score and asks a human
rather than applying the top match. The defensible workflow is to normalise
automatically only above a high confidence threshold, review the middle band by
hand, and report the threshold — an unreported normalisation step makes a
frequency table unreproducible.

</details>